In [1]:

# circular buffer



import numpy as np

import tiledb
from tiledb.ml.models.pytorch import PyTorchTileDBModel

from rosbags.rosbag2 import Reader
from rosbags.serde import deserialize_cdr

import cv2

from cv_bridge import CvBridge, CvBridgeError

import os
import shutil


from rclpy.time import Time

# import ros2_numpy as rnp
## pip install ros2-numpy




In [ ]:


import builtin_interfaces.msg

from math import e

with Reader('/data/INS/small/') as reader:
    
    for conn in reader.connections:
        print("topic: ", conn.topic, ", type: ", conn.msgtype, ", count: ", conn.msgcount)

    # for connection, timestamp, rawdata in reader.messages():
    #     if connection.topic == '/oak1_left/image_raw':    
    #         msg = deserialize_cdr(rawdata, connection.msgtype)


topic:  /novatel/oem7/bestutm , type:  novatel_oem7_msgs/msg/BESTUTM , count:  2089
topic:  /novatel/oem7/odom , type:  nav_msgs/msg/Odometry , count:  96626
topic:  /novatel/oem7/bestpos , type:  novatel_oem7_msgs/msg/BESTPOS , count:  15154
topic:  /events/write_split , type:  rosbag2_interfaces/msg/WriteSplitEvent , count:  0
topic:  /novatel/oem7/odom_origin , type:  nav_msgs/msg/Odometry , count:  0
topic:  /novatel/oem7/bestvel , type:  novatel_oem7_msgs/msg/BESTVEL , count:  15333
topic:  /disk_usage , type:  std_msgs/msg/String , count:  2089
topic:  /client_count , type:  std_msgs/msg/Int32 , count:  6
topic:  /blackfly_5/meta , type:  flir_camera_msgs/msg/ImageMetaData , count:  0
topic:  /anello/odom , type:  nav_msgs/msg/Odometry , count:  208769
topic:  /novatel/oem7/time , type:  novatel_oem7_msgs/msg/TIME , count:  2089
topic:  /blackfly_5/camera_info_ext , type:  flir_camera_msgs/msg/CameraInfoExt , count:  0
topic:  /blackfly_4/camera_info , type:  sensor_msgs/msg/Came

In [ ]:

import builtin_interfaces.msg

from math import e

bridge = CvBridge()
px_rows = 360
px_cols = 640
channels = 3

image_buffer = np.zeros((10, 360, 640, 3), dtype=int)
time_buffer = np.zeros((10), dtype=np.int64)

with Reader('/data/notebooks/data/oak/') as reader:

    topics = []
    types = []
    counts = []
    
    for conn in reader.connections:
        topics.append(conn.topic)
        types.append(conn.msgtype)    
        print("topic: ", conn.topic, ", type: ", conn.msgtype, ", count: ", conn.msgcount)
        counts.append(conn.msgcount)

    for connection, timestamp, rawdata in reader.messages():
        if connection.topic == '/oak1_left/image_raw':    
            msg = deserialize_cdr(rawdata, connection.msgtype)
            np_arr = np.frombuffer(msg.data, np.uint8)
            cv_image = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
            frame = bridge.imgmsg_to_cv2(msg, "passthrough")
            im = np.asarray(frame)

            image_buffer[:-1] = image_buffer[1:]
            image_buffer[-1] = im

            ts = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e+9
            
            # TODO: buffer for timestamps
            time_buffer[:-1] = time_buffer[1:]
            time_buffer[-1] = ts
            
            # print(circular_buffer)
            break


topic:  /events/write_split , type:  rosbag2_interfaces/msg/WriteSplitEvent , count:  0
topic:  /oak1_left/image_raw , type:  sensor_msgs/msg/Image , count:  1039
topic:  /oak1_right/image_raw , type:  sensor_msgs/msg/Image , count:  1038
topic:  /parameter_events , type:  rcl_interfaces/msg/ParameterEvent , count:  0
topic:  /rosout , type:  rcl_interfaces/msg/Log , count:  9
1697248867
159353392
1697248867.1593535


In [97]:


from pathlib import Path
from rosbags.highlevel import AnyReader


orient_buffer = np.zeros((10, 4), dtype=np.float64)
orient_cov_buffer = np.zeros((10, 9), dtype=np.float64)

ang_vel_buffer = np.zeros((10, 3), dtype=np.float64)
ang_vel_cov_buffer = np.zeros((10, 9), dtype=np.float64)

lin_acc_buffer = np.zeros((10, 3), dtype=np.float64)
lin_acc_cov_buffer = np.zeros((10, 9), dtype=np.float64)

time_buffer = np.zeros((10), dtype=np.int64)


with AnyReader([Path('/external_files/notebooks/data/data.bag')]) as reader:
    # for connection in reader.connections:
    #     print(connection.topic, connection.msgtype)

    connections = [x for x in reader.connections if x.topic == '/imu/data_raw']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
        msg = reader.deserialize(rawdata, connection.msgtype)

        print("orient     : ", msg.orientation)
        print("orient  cov: ", msg.orientation_covariance)
        print("ang vel    : ", msg.angular_velocity)
        print("ang vel cov: ", msg.angular_velocity_covariance)
        print("lin acc    : ", msg.linear_acceleration)
        print("lin acc cov: ", msg.linear_acceleration_covariance)

        orient_buffer[:-1] = orient_buffer[1:]
        orient_buffer[-1] = np.array([msg.orientation.x, msg.orientation.y, msg.orientation.z, msg.orientation.w])

        orient_cov_buffer[:-1] = orient_cov_buffer[1:]
        orient_cov_buffer[-1] = msg.orientation_covariance

        ###
        ang_vel_buffer[:-1] = ang_vel_buffer[1:]
        ang_vel_buffer[-1] = np.array([msg.angular_velocity.x, msg.angular_velocity.y, msg.angular_velocity.z])

        ang_vel_cov_buffer[:-1] = ang_vel_cov_buffer[1:]
        ang_vel_cov_buffer[-1] = msg.angular_velocity_covariance
        ###

        ###
        lin_acc_buffer[:-1] = lin_acc_buffer[1:]
        lin_acc_buffer[-1] = np.array([msg.linear_acceleration.x, msg.linear_acceleration.y, msg.linear_acceleration.z])

        lin_acc_cov_buffer[:-1] = lin_acc_cov_buffer[1:]
        lin_acc_cov_buffer[-1] = msg.linear_acceleration_covariance
        ###
        
        ts = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e+9
        time_buffer[:-1] = time_buffer[1:]
        time_buffer[-1] = ts
            

        break


# /imu/data_raw sensor_msgs/msg/Imu
# /gps/gps gps_common/msg/GPSFix


# camera
# lidar?
# imu?
# gps?

# the buffer object should handle any type without code ducplication



orient     :  geometry_msgs__msg__Quaternion(x=0.0, y=0.0, z=0.0, w=0.0, __msgtype__='geometry_msgs/msg/Quaternion')
orient  cov:  [-1.  0.  0.  0.  0.  0.  0.  0.  0.]
ang vel    :  geometry_msgs__msg__Vector3(x=0.0001805368810892105, y=2.9802322387695312e-05, z=-0.00010881340131163597, __msgtype__='geometry_msgs/msg/Vector3')
ang vel cov:  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
lin acc    :  geometry_msgs__msg__Vector3(x=0.2637051045894623, y=0.07881093770265579, z=-9.803738631308079, __msgtype__='geometry_msgs/msg/Vector3')
lin acc cov:  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
